# Mental Health Risk Prediction Using Social Media and Psychological Indicators

Complete beginner-friendly ML classification workflow.


## 1. Data Understanding
Load the dataset, inspect records, schema, statistics, and column meanings.


In [ ]:
from pathlib import Path
import joblib
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

sns.set_theme(style='whitegrid')
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATHS = [PROJECT_DIR / 'data' / 'mental_health_trends.csv', PROJECT_DIR.parent / 'projects' / 'mental_health_trends' / 'dataset' / 'mental_health_trends.csv']
DATA_PATH = next(path for path in DATA_PATHS if path.exists())
df = pd.read_csv(DATA_PATH)
df.head()


In [ ]:
df.info()
df.describe(include='all')


### Column Explanation
- `user_id`: anonymous user identifier.
- `year`: record year.
- `country`: user country.
- `age_group`: age segment.
- `gender`: self-reported gender.
- `platform`: social media platform.
- `anxiety_score`, `depression_score`, `stress_level`, `loneliness_index`, `self_esteem_score`: numeric psychological indicators.
- `therapy_access`, `medication_usage`: support/treatment indicators.
- `mental_health_risk`: target class.


## 2. Data Preprocessing


In [ ]:
print(df.isna().sum())
df_clean = df.copy()
for col in df_clean.select_dtypes(include='number').columns:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())
for col in df_clean.select_dtypes(exclude='number').columns:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
df_clean = df_clean.drop_duplicates()
label_encoder = LabelEncoder()
df_clean['mental_health_risk_encoded'] = label_encoder.fit_transform(df_clean['mental_health_risk'])
label_encoder.classes_


Feature selection uses the required five numeric psychological indicators because they are direct risk predictors and are simple to deploy in a web form.


## 3. Exploratory Data Analysis


In [ ]:
features = ['anxiety_score','depression_score','stress_level','loneliness_index','self_esteem_score']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(df['anxiety_score'], kde=True, ax=axes[0,0])
sns.histplot(df['depression_score'], kde=True, ax=axes[0,1])
sns.boxplot(y=df['stress_level'], ax=axes[1,0])
sns.countplot(data=df, x='mental_health_risk', ax=axes[1,1])
plt.tight_layout()


In [ ]:
sns.scatterplot(data=df, x='anxiety_score', y='depression_score', hue='mental_health_risk')
plt.title('Anxiety vs Depression')
plt.show()
sns.heatmap(df[features].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x='gender', ax=axes[0])
sns.countplot(data=df, x='platform', ax=axes[1])
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()


## 4-9. Classification, Evaluation, Cross Validation, and Model Comparison


In [ ]:
X = df_clean[features]
y = df_clean['mental_health_risk_encoded']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
models = {
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000))]),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
}
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, average='weighted', zero_division=0),
        'Recall': recall_score(y_test, pred, average='weighted', zero_division=0),
        'F1 Score': f1_score(y_test, pred, average='weighted', zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, proba, multi_class='ovr', average='weighted')
    })
    print(name)
    print(classification_report(y_test, pred, target_names=label_encoder.classes_))
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
    plt.title(f'{name} Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
comparison = pd.DataFrame(results).sort_values('F1 Score', ascending=False)
comparison


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    print(f'{name}: {scores.mean():.4f} +/- {scores.std():.4f}')


## 10. Feature Importance


In [ ]:
rf = models['Random Forest']
importance = pd.DataFrame({'Feature': features, 'Importance': rf.feature_importances_}).sort_values('Importance', ascending=False)
sns.barplot(data=importance, x='Importance', y='Feature')
plt.title('Random Forest Feature Importance')
plt.show()
importance


## 11. Responsible AI and Ethics
Bias can appear across gender, country, platform, age group, therapy access, and medication usage. Mental health data is sensitive and may be incomplete, self-reported, synthetic, or non-clinical. This model should support education and awareness only. It must not replace professional diagnosis, therapy, emergency help, or clinical judgment.


## 13-14. Conclusion and Deployment Preparation


In [ ]:
best_name = comparison.iloc[0]['Model']
best_model = models[best_name]
Path('models').mkdir(exist_ok=True)
joblib.dump(best_model, 'models/model.pkl')
joblib.dump(label_encoder, 'models/label_encoder.pkl')
print(f'Best model: {best_name}')
print('Saved models/model.pkl and models/label_encoder.pkl')
